# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

In [1]:
import os
import importlib
import numpy as np
import alc

importlib.reload(alc)
np.set_printoptions(precision=4, suppress=True)

# rutas (notebook en la raíz del repo)
PATH_CORNERS = os.path.join("calib", "coplanar", "detected_corners.txt")
N_IMG, N_PTS = 10, 77

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


In [21]:
# proyecciones detectadas: 1 fila por imagen, esquinas intercaladas (u,v)
detected = np.loadtxt(PATH_CORNERS)              # (10, 154)
print("detected:", detected.shape)               # sanity del formato

# puntos del tablero: analíticos, fila-mayor (X=30i, Y=30j, Z=0) en mm
n = np.arange(N_PTS)
objPts = np.column_stack([30.0*(n % 11), 30.0*(n // 11), np.zeros(N_PTS)])
print("objPts[0], objPts[11], objPts[76]:", objPts[0], objPts[11], objPts[76])

X, Y = objPts[:, 0], objPts[:, 1]        # (usadas por el QA de reproyección y los plots

detected: (10, 154)
objPts[0], objPts[11], objPts[76]: [0. 0. 0.] [ 0. 30.  0.] [300. 180.   0.]


### Calibración (Zhang 2000, caso coplanar)

Cada imagen aporta la homografía $H_j$ (solución de mínimos cuadrados
homogéneos, ec. 8) y las restricciones

$$ \mathbf{c}_1^{j\top} B\, \mathbf{c}_2^j = 0, \qquad
   \mathbf{c}_1^{j\top} B\, \mathbf{c}_1^j \;=\; \mathbf{c}_2^{j\top} B\, \mathbf{c}_2^j $$

con $B = K^{-\top}K^{-1}$ simétrica ⇒ 6 incógnitas $\mathbf b=(B_{11},B_{12},B_{22},B_{13},B_{23},B_{33})^\top$.
Armando $A\in\mathbb{R}^{2N\times 6}$ y tomando el **vector nulo** de $A$ (norma 1)
obtenemos $\mathbf b$; el signo se elige de modo que $B$ sea definida positiva,
y $K = (L^\top)^{-1}$ con $B = LL^\top$ (Cholesky), normalizada con $K_{33}=1$.

a) Obtener la matriz $\mathbf{K}$

In [3]:
# 1. Homografía por imagen: DLT coplanar (ec. 11 + ec. 8)
Hlst = []
for k in range(N_IMG):
    imgPts = detected[k].reshape(N_PTS, 2)
    Q  = alc.obtenerQ2d(imgPts, objPts)            # (154, 9)
    H  = alc.resolverDLT2D(Q)                      # (3, 3), ‖h‖=1 
    Hlst.append(H)

# 2. Matriz A de restricciones de Zhang (ec. 13): 2 filas por vista
A = alc.armarA(Hlst)                               # (20, 6)

# 3. K: núcleo de A → b → B def. positiva → Cholesky → K (K33=1)
K = alc.calcularMatrizK2D(A)
print("K:\n", K)

K:
 [[742.3299  -0.6789 520.4803]
 [  0.     753.564  299.2673]
 [  0.       0.       1.    ]]


b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

In [5]:
# Pose (R1, t1) de la cámara respecto del tablero en la captura Im_R_1.png;
# t viaja en mm (λ = 1/‖K⁻¹c₁‖ re-escala la normalización arbitraria del
# vector nulo de la SVD a las unidades de las esquinas del tablero).
# El signo de H es arbitrario: si el DLT devuelve −H, la pose sale como la
# variante alternativa válida (R' = R·diag(−1,−1,1)) — ver markdown de abajo.
Rj1, tj1 = alc.obtenerPose2D(Hlst[0], K)
Rj10, tj10 = alc.obtenerPose2D(Hlst[9], K)
print(f'======== Pose Im_R_1 ======\n Rj1 \n{Rj1}\n tj1\n{tj1}')
print(f'======== Pose Im_R_10 ======\n Rj10 \n{Rj10}\n tj10\n{tj10}')



======== Pose Im_R_1 ======
 Rj1 
[[-0.9995  0.013  -0.0276]
 [-0.0121 -0.9994  0.0078]
 [-0.0275  0.0082  0.9991]]
 tj1
[ 109.2896  141.8554 -548.2162]
======== Pose Im_R_10 ======
 Rj10 
[[-0.9974 -0.0682 -0.025 ]
 [ 0.0632 -0.9856  0.157 ]
 [-0.0353  0.155   0.9874]]
 tj10
[  96.5027  154.4873 -538.9271]


### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---